# 54 — Mini project: bộ chuẩn bị dữ liệu tái sử dụng

**Sản phẩm của nhóm cơ bản.** Ba notebook `51`–`53` dạy ba nhóm kỹ năng rời
rạc. Notebook này ghép chúng thành một thứ bạn thật sự dán vào dự án của mình:
một hàm **nhận mã chứng khoán, trả về frame sạch, đúng kiểu, đủ cột phái sinh,
và nhẹ hơn nhiều lần**.

Đây là tầng code mọi phân tích đều cần và ai cũng viết lại từ đầu mỗi lần.

| Bước | Dùng kỹ năng từ |
|---|---|
| Kiểm tra chất lượng dữ liệu | `51` — `dtypes`, `isna`, `duplicated` |
| Tối ưu kiểu dữ liệu và bộ nhớ | `51` — `category`, downcast, `memory_usage(deep=True)` |
| Thêm cột phái sinh an toàn | `52` — CoW, `.loc`, không sửa tại chỗ |
| Tính cột vectorised | `53` — `assign`, `np.select`, `pd.cut` |
| Lưu và nạp lại | ⚠️ chỗ CSV quên hết kiểu dữ liệu |

In [1]:
import sys
import time
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd

import finlens
from finlens_examples import hom_nay, lui_ngay

client = finlens.client()
HOM_NAY = hom_nay(client)

THU_MUC_RA = GOC / "output"
THU_MUC_RA.mkdir(exist_ok=True)

print(f"pandas {pd.__version__} · finlens {finlens.build_info()['version']}")

pandas 3.0.5 · finlens 1.3.0


## 1 · Vấn đề

Frame thô từ API dùng nhiều bộ nhớ hơn mức cần, và không có cột phái sinh nào.
Với ba mã thì không sao. Với toàn sàn nhiều năm thì đó là khác biệt giữa chạy
được và hết RAM.

In [2]:
MA = client.meta.symbols(exchange="HOSE", kind="stock")["symbol"].tolist()[:200]

t0 = time.perf_counter()
tho = client.eod.stock.ohlcv(MA, start=lui_ngay(HOM_NAY, nam=2))
t_tai = time.perf_counter() - t0

print(f"Tải {tho['symbol'].nunique()} mã · {len(tho):,} dòng trong {t_tai:.1f} giây")
print(f"Bộ nhớ: {tho.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print()
print(tho.dtypes.to_string())

Tải 199 mã · 97,625 dòng trong 18.2 giây
Bộ nhớ: 9.3 MB

symbol            string
date      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume           float64


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


## 2 · Bước một — kiểm tra chất lượng

Luôn chạy trước khi tính bất cứ thứ gì. Sáu câu hỏi, mỗi câu một dòng code.

In [3]:
def kiem_tra_chat_luong(df: pd.DataFrame, *, khoa: tuple[str, ...] = ("symbol", "date")) -> pd.DataFrame:
    """Báo cáo chất lượng một frame giá — chạy trước mọi phép tính.

    Trả về frame một dòng để ghép được nhiều lần kiểm tra lại với nhau, thay vì
    in ra rồi mất.
    """
    so_am = df.select_dtypes("number").lt(0).sum().sum()
    return pd.DataFrame(
        [
            {
                "dòng": len(df),
                "mã": df["symbol"].nunique() if "symbol" in df else np.nan,
                "phiên đầu": df["date"].min() if "date" in df else pd.NaT,
                "phiên cuối": df["date"].max() if "date" in df else pd.NaT,
                "ô thiếu": int(df.isna().sum().sum()),
                "dòng trùng khoá": int(df.duplicated(subset=list(khoa)).sum()),
                "giá trị âm": int(so_am),
                "MB": round(df.memory_usage(deep=True).sum() / 1024**2, 1),
            }
        ]
    )


bao_cao_truoc = kiem_tra_chat_luong(tho)
bao_cao_truoc

,dòng,mã,phiên đầu,phiên cuối,ô thiếu,dòng trùng khoá,giá trị âm,MB
0,97625,199,2024-08-12,2026-08-12,0,0,0,9.3


⚠️ **`duplicated(subset=…)` cần nêu rõ khoá.** Không có `subset`, pandas so cả
hàng — và hai phiên khác nhau tình cờ cùng giá sẽ *không* bị coi là trùng, còn
một mã bị tải hai lần với giá đã điều chỉnh khác nhau thì cũng không bị bắt.
Khoá thật của dữ liệu này là cặp `(symbol, date)`.

In [4]:
# Chứng minh: tạo một bản trùng khoá nhưng khác giá trị
gia_lap = pd.concat([tho.head(3), tho.head(3).assign(close=lambda d: d["close"] + 1)])
print(f"duplicated() không subset : {gia_lap.duplicated().sum()} dòng trùng")
print(f"duplicated(subset=khoá)   : {gia_lap.duplicated(subset=['symbol', 'date']).sum()} dòng trùng  ← đúng")

duplicated() không subset : 0 dòng trùng
duplicated(subset=khoá)   : 3 dòng trùng  ← đúng


### Kiểm tra riêng cho dữ liệu giá

Ba khẳng định mà mọi frame OHLCV phải thoả. Nếu sai thì dữ liệu hỏng, không
phải phép tính của bạn hỏng.

In [5]:
def kiem_tra_ohlcv(df: pd.DataFrame) -> pd.DataFrame:
    """Ba bất biến của một thanh giá. Trả về các dòng VI PHẠM."""
    vi_pham = pd.DataFrame(
        {
            "high < low": df["high"] < df["low"],
            "close ngoài [low, high]": (df["close"] < df["low"]) | (df["close"] > df["high"]),
            "open ngoài [low, high]": (df["open"] < df["low"]) | (df["open"] > df["high"]),
            "volume âm": df["volume"] < 0,
        }
    )
    return vi_pham.sum().rename("số dòng vi phạm").to_frame()


kiem_tra_ohlcv(tho)

,số dòng vi phạm
high < low,0
"close ngoài [low, high]",0
"open ngoài [low, high]",0
volume âm,0


## 3 · Bước hai — tối ưu kiểu dữ liệu

Hai phép, và cả hai đến thẳng từ notebook `51`.

In [6]:
def toi_uu_kieu(df: pd.DataFrame) -> pd.DataFrame:
    """Thu nhỏ frame mà không mất thông tin. Trả về frame mới.

    - `symbol` → `category`: vài trăm giá trị lặp lại hàng trăm nghìn lần
    - cột giá → `float32`: giá cổ phiếu Việt Nam tối đa 5 chữ số, thừa sức
    - `volume` → `int64`: khối lượng vốn là số nguyên, không cần phần thập phân

    ⚠️ Trả về frame MỚI thay vì sửa tại chỗ — quy tắc từ notebook `52`.
    """
    kieu_moi: dict[str, str] = {}
    if "symbol" in df.columns:
        kieu_moi["symbol"] = "category"
    for c in ("open", "high", "low", "close"):
        if c in df.columns:
            kieu_moi[c] = "float32"
    if "volume" in df.columns and df["volume"].notna().all():
        kieu_moi["volume"] = "int64"
    return df.astype(kieu_moi)


toi_uu = toi_uu_kieu(tho)

so_sanh_bo_nho = pd.DataFrame(
    {
        "thô (MB)": (tho.memory_usage(deep=True) / 1024**2).round(2),
        "tối ưu (MB)": (toi_uu.memory_usage(deep=True) / 1024**2).round(2),
    }
)
so_sanh_bo_nho["giảm"] = (
    (1 - so_sanh_bo_nho["tối ưu (MB)"] / so_sanh_bo_nho["thô (MB)"].replace(0, np.nan)) * 100
).round(0).fillna(0).astype(int).astype(str) + "%"
so_sanh_bo_nho

,thô (MB),tối ưu (MB),giảm
Index,0.00,0.00,0%
symbol,4.84,0.20,96%
date,0.74,0.74,0%
open,0.74,0.37,50%
high,0.74,0.37,50%
low,0.74,0.37,50%
close,0.74,0.37,50%
volume,0.74,0.74,0%


In [7]:
tong_truoc = tho.memory_usage(deep=True).sum() / 1024**2
tong_sau = toi_uu.memory_usage(deep=True).sum() / 1024**2
print(f"Tổng: {tong_truoc:.1f} MB → {tong_sau:.1f} MB   (nhẹ hơn {tong_truoc / tong_sau:.1f} lần)")

Tổng: 9.3 MB → 3.2 MB   (nhẹ hơn 2.9 lần)


### ⚠️ `float32` mất bao nhiêu độ chính xác?

Đừng tin lời tôi — đo.

In [8]:
lech = (tho["close"].astype("float32").astype("float64") - tho["close"]).abs()
print(f"Sai lệch tuyệt đối lớn nhất: {lech.max():.2e} nghìn VND = {lech.max() * 1000:.6f} VND")
print(f"Giá cao nhất trong frame   : {tho['close'].max():,.2f} nghìn VND")
print(f"Bước giá nhỏ nhất của HOSE : 0,01 nghìn VND = 10 VND")
print()
print(f"→ sai lệch nhỏ hơn bước giá {0.01 / lech.max():,.0f} lần. An toàn cho giá cổ phiếu Việt Nam.")

Sai lệch tuyệt đối lớn nhất: 7.32e-06 nghìn VND = 0.007324 VND
Giá cao nhất trong frame   : 178.58 nghìn VND
Bước giá nhỏ nhất của HOSE : 0,01 nghìn VND = 10 VND

→ sai lệch nhỏ hơn bước giá 1,365 lần. An toàn cho giá cổ phiếu Việt Nam.


**Nhưng đừng dùng `float32` cho giá trị giao dịch.** GTGD lên tới hàng nghìn tỷ
đồng, và `float32` chỉ giữ được khoảng 7 chữ số có nghĩa:

In [9]:
gtgd_mau = 1_234_567_890_123.0  # 1.234 nghìn tỷ đồng

# ⚠️ Ép về float64 TRƯỚC khi trừ. numpy 2 coi số float của Python là "yếu"
# (NEP 50), nên phép trừ float32 với float thường vẫn ra float32 — sai lệch tự
# làm tròn về 0, và phép đo xoá mất chính thứ nó định đo.
sai_lech = float(np.float32(gtgd_mau)) - gtgd_mau

print(f"float64: {gtgd_mau:,.0f}")
print(f"float32: {float(np.float32(gtgd_mau)):,.0f}")
print(f"→ sai {abs(sai_lech):,.0f} đồng")
print()
print(f"Trừ ngay trong float32 thì ra: {abs(np.float32(gtgd_mau) - gtgd_mau):,.0f} đồng  ← phép đo tự lừa mình")

float64: 1,234,567,890,123
float32: 1,234,567,954,432
→ sai 64,309 đồng

Trừ ngay trong float32 thì ra: 0 đồng  ← phép đo tự lừa mình


## 4 · Bước ba — thêm cột phái sinh

Đây là chỗ notebook `52` và `53` gặp nhau: mọi phép tính theo mã **phải qua
`groupby`**, và toàn bộ hàm **trả về frame mới** chứ không sửa tại chỗ.

In [10]:
def them_cot_phai_sinh(df: pd.DataFrame) -> pd.DataFrame:
    """Thêm các cột phân tích thường dùng. Trả về frame mới.

    ⚠️ `pct_change` và `shift` đi qua `groupby("symbol")` — nếu không, dòng đầu
    của mỗi mã sẽ lấy giá cuối của mã đứng trước làm mốc. Cùng lỗi rò rỉ ranh
    giới nhóm đã đo ở notebook `31`.
    """
    d = df.sort_values(["symbol", "date"]).copy()
    theo_ma = d.groupby("symbol", observed=True)

    return d.assign(
        # Giá tính bằng nghìn VND → nhân 1.000 ra VND
        gtgd=lambda x: x["close"].astype("float64") * x["volume"] * 1_000,
        ls=lambda x: theo_ma["close"].pct_change() * 100,
        bien_do=lambda x: (x["high"] - x["low"]) / x["low"] * 100,
        kl_bq_20=lambda x: theo_ma["volume"].transform(lambda s: s.rolling(20, min_periods=20).mean()),
        so_lan_kl=lambda x: x["volume"] / x["kl_bq_20"],
        huong=lambda x: np.select(
            [x["ls"] > 0, x["ls"] < 0], ["tăng", "giảm"], default="đứng giá"
        ),
        muc_bien_dong=lambda x: pd.cut(
            x["bien_do"],
            bins=[-np.inf, 1, 3, 5, np.inf],
            labels=["thấp", "vừa", "cao", "rất cao"],
        ),
    )


day_du = them_cot_phai_sinh(toi_uu)

print(f"{toi_uu.shape[1]} cột → {day_du.shape[1]} cột")
print(f"Thêm: {[c for c in day_du.columns if c not in toi_uu.columns]}")
day_du.head(3)

7 cột → 14 cột
Thêm: ['gtgd', 'ls', 'bien_do', 'kl_bq_20', 'so_lan_kl', 'huong', 'muc_bien_dong']


,symbol,date,open,high,low,close,volume,gtgd,ls,bien_do,kl_bq_20,so_lan_kl,huong,muc_bien_dong
0,AAA,2024-08-12,9.57,9.66,9.48,9.61,4558000,4.380238e+10,NaN,1.898737,NaN,NaN,đứng giá,vừa
1,AAA,2024-08-13,9.66,9.75,9.57,9.66,3049900,2.946203e+10,0.520289,1.880881,NaN,NaN,tăng,vừa
2,AAA,2024-08-14,9.75,9.75,9.57,9.57,3149400,3.013976e+10,-0.931680,1.880881,NaN,NaN,giảm,vừa


### Kiểm chứng ranh giới nhóm

Đừng tin là `groupby` đã chạy đúng — khẳng định nó.

In [11]:
dau_moi_ma = day_du.groupby("symbol", observed=True).head(1)
print(f"Dòng đầu của mỗi mã có ls = NaN: {dau_moi_ma['ls'].isna().sum()}/{len(dau_moi_ma)} mã")
assert dau_moi_ma["ls"].isna().all(), "Rò rỉ ranh giới nhóm! pct_change không qua groupby"
print("✓ Không có rò rỉ giữa các mã")

Dòng đầu của mỗi mã có ls = NaN: 199/199 mã
✓ Không có rò rỉ giữa các mã


In [12]:
# Đối chiếu với cách SAI để thấy khác biệt
sai = toi_uu.sort_values(["symbol", "date"]).assign(ls_sai=lambda d: d["close"].pct_change() * 100)
dau_sai = sai.groupby("symbol", observed=True).head(1)
print(f"Không qua groupby → chỉ {dau_sai['ls_sai'].isna().sum()}/{len(dau_sai)} mã có NaN ở dòng đầu")
print(f"→ {len(dau_sai) - dau_sai['ls_sai'].isna().sum()} mã lấy giá của mã khác làm mốc, cho ra lợi suất bịa")
print()
lech_lon = dau_sai["ls_sai"].abs().max()
print(f"Lợi suất bịa lớn nhất: {lech_lon:,.1f}%   ← không có mã nào tăng ngần ấy trong một phiên")

Không qua groupby → chỉ 1/199 mã có NaN ở dòng đầu
→ 198 mã lấy giá của mã khác làm mốc, cho ra lợi suất bịa

Lợi suất bịa lớn nhất: 2,881.1%   ← không có mã nào tăng ngần ấy trong một phiên


## 5 · Bước bốn — đóng gói lại

Bốn hàm trên thành một. Đây là thứ bạn dán vào dự án.

In [13]:
def chuan_bi_du_lieu(
    client,
    ma: list[str],
    *,
    start: str,
    end: str | None = None,
    toi_uu_bo_nho: bool = True,
    kiem_tra: bool = True,
) -> pd.DataFrame:
    """Tải, kiểm tra, tối ưu và làm giàu dữ liệu giá — một lời gọi.

    Trả về frame mới, đã sắp theo ``(symbol, date)``, kiểu dữ liệu tối ưu, kèm
    các cột phái sinh thường dùng.

    ``kiem_tra=True`` dừng bằng ``AssertionError`` nếu dữ liệu vi phạm bất biến
    OHLCV — thà hỏng ồn ào còn hơn tính toán trên dữ liệu hỏng.
    """
    df = client.eod.stock.ohlcv(ma, start=start, end=end)

    if df.empty:
        raise ValueError(f"Không có dữ liệu cho {len(ma)} mã trong khoảng đã cho")

    if df.attrs.get("finlens", {}).get("truncated"):
        raise RuntimeError("Kết quả bị cắt vì max_rows — chia nhỏ danh sách mã hoặc thu hẹp khoảng thời gian")

    if kiem_tra:
        vi_pham = kiem_tra_ohlcv(df)
        assert vi_pham["số dòng vi phạm"].sum() == 0, f"Dữ liệu vi phạm bất biến OHLCV:\n{vi_pham}"
        trung = df.duplicated(subset=["symbol", "date"]).sum()
        assert trung == 0, f"{trung} dòng trùng khoá (symbol, date)"

    # ⚠️ Đọc đơn vị TRƯỚC mọi biến đổi — attrs không sống sót qua nhiều phép
    don_vi = dict(df.attrs.get("finlens", {}).get("units", {}))

    if toi_uu_bo_nho:
        df = toi_uu_kieu(df)
    df = them_cot_phai_sinh(df)

    df.attrs["don_vi_goc"] = don_vi
    return df.reset_index(drop=True)


t0 = time.perf_counter()
san_sang = chuan_bi_du_lieu(client, MA, start=lui_ngay(HOM_NAY, nam=2))
t_chuan_bi = time.perf_counter() - t0

print(f"Xong trong {t_chuan_bi:.1f} giây")
print(f"{len(san_sang):,} dòng × {san_sang.shape[1]} cột · {san_sang.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Đơn vị gốc đã giữ lại: {san_sang.attrs['don_vi_goc']}")

Xong trong 17.2 giây
97,625 dòng × 14 cột · 13.3 MB
Đơn vị gốc đã giữ lại: {'symbol': None, 'date': None, 'open': 'kVND', 'high': 'kVND', 'low': 'kVND', 'close': 'kVND', 'volume': 'share'}


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


### Nó bắt lỗi thật không?

Một hàm kiểm tra chưa từng thất bại là một hàm chưa được kiểm tra.

In [14]:
hong = tho.head(100).copy()
hong.loc[hong.index[0], "high"] = 0.0  # high < low → vi phạm bất biến

vi_pham = kiem_tra_ohlcv(hong)
print("Trên dữ liệu đã bẻ hỏng:")
print(vi_pham[vi_pham["số dòng vi phạm"] > 0].to_string())

try:
    assert vi_pham["số dòng vi phạm"].sum() == 0
except AssertionError:
    print("\n✓ AssertionError được ném đúng như thiết kế")

Trên dữ liệu đã bẻ hỏng:
                         số dòng vi phạm
high < low                             1
close ngoài [low, high]                1
open ngoài [low, high]                 1

✓ AssertionError được ném đúng như thiết kế


## 6 · Bước năm — lưu và nạp lại

⚠️ Đây là chỗ công sức tối ưu kiểu dữ liệu ở bước hai **bốc hơi**.

In [15]:
tep_csv = THU_MUC_RA / "du_lieu_chuan_bi.csv"
san_sang.to_csv(tep_csv, index=False)

nap_lai = pd.read_csv(tep_csv)

so_kieu = pd.DataFrame(
    {"trước khi lưu": san_sang.dtypes.astype(str), "sau khi nạp CSV": nap_lai.dtypes.astype(str)}
)
so_kieu["giữ nguyên"] = np.where(so_kieu["trước khi lưu"] == so_kieu["sau khi nạp CSV"], "✓", "✗")
so_kieu

,trước khi lưu,sau khi nạp CSV,giữ nguyên
symbol,category,str,✗
date,datetime64[ns],str,✗
open,float32,float64,✗
high,float32,float64,✗
low,float32,float64,✗
close,float32,float64,✗
volume,int64,int64,✓
gtgd,float64,float64,✓
ls,float32,float64,✗
bien_do,float32,float64,✗


In [16]:
print(f"Bộ nhớ trước khi lưu : {san_sang.memory_usage(deep=True).sum() / 1024**2:>6.1f} MB")
print(f"Bộ nhớ sau khi nạp   : {nap_lai.memory_usage(deep=True).sum() / 1024**2:>6.1f} MB")
print(f"→ phình lên {nap_lai.memory_usage(deep=True).sum() / san_sang.memory_usage(deep=True).sum():.1f} lần")
print()
print(f"Cột date sau khi nạp: {nap_lai['date'].dtype}   ← đã thành chuỗi, không còn là ngày tháng")
print(f"Cột muc_bien_dong  : {nap_lai['muc_bien_dong'].dtype}   ← category thành chuỗi, mất thứ tự")

Bộ nhớ trước khi lưu :   13.3 MB
Bộ nhớ sau khi nạp   :   31.3 MB
→ phình lên 2.4 lần

Cột date sau khi nạp: str   ← đã thành chuỗi, không còn là ngày tháng
Cột muc_bien_dong  : str   ← category thành chuỗi, mất thứ tự


**CSV là văn bản thuần — nó không có chỗ nào để ghi kiểu dữ liệu.** Mọi thứ bạn
vừa tối ưu bị quên sạch, và `date` quay về dạng chuỗi.

Ba cách xử lý, theo thứ tự nên chọn:

In [17]:
print("1. Dùng định dạng có lưu kiểu — parquet (cần `pip install pyarrow`):")
print("     df.to_parquet('du_lieu.parquet')")
print("     pd.read_parquet('du_lieu.parquet')   ← dtype giữ nguyên hoàn toàn")
print()
print("2. Nếu buộc dùng CSV, lưu kèm bản đồ kiểu và áp lại lúc nạp")
print()
print("3. Đơn giản nhất khi chỉ cần date: parse_dates=[...] lúc đọc")

1. Dùng định dạng có lưu kiểu — parquet (cần `pip install pyarrow`):
     df.to_parquet('du_lieu.parquet')
     pd.read_parquet('du_lieu.parquet')   ← dtype giữ nguyên hoàn toàn

2. Nếu buộc dùng CSV, lưu kèm bản đồ kiểu và áp lại lúc nạp

3. Đơn giản nhất khi chỉ cần date: parse_dates=[...] lúc đọc


In [18]:
# Cách 2 — tự lưu bản đồ kiểu bên cạnh
import json

ban_do_kieu = {c: str(t) for c, t in san_sang.dtypes.items()}
(THU_MUC_RA / "du_lieu_chuan_bi.dtypes.json").write_text(
    json.dumps(ban_do_kieu, ensure_ascii=False, indent=2), encoding="utf-8"
)


def nap_kem_kieu(tep_csv: Path) -> pd.DataFrame:
    """Nạp CSV rồi áp lại bản đồ kiểu đã lưu bên cạnh."""
    ban_do = json.loads(tep_csv.with_suffix(".dtypes.json").read_text(encoding="utf-8"))
    ngay = [c for c, t in ban_do.items() if "datetime" in t]
    khac = {c: t for c, t in ban_do.items() if "datetime" not in t}
    return pd.read_csv(tep_csv, parse_dates=ngay).astype(khac)


phuc_hoi = nap_kem_kieu(tep_csv)

khop = (phuc_hoi.dtypes.astype(str) == san_sang.dtypes.astype(str)).sum()
print(f"Sau khi áp lại bản đồ kiểu: {khop}/{len(san_sang.dtypes)} cột khớp kiểu gốc")
print(f"Bộ nhớ: {phuc_hoi.memory_usage(deep=True).sum() / 1024**2:.1f} MB "
      f"(gốc {san_sang.memory_usage(deep=True).sum() / 1024**2:.1f} MB)")

Sau khi áp lại bản đồ kiểu: 13/14 cột khớp kiểu gốc
Bộ nhớ: 13.3 MB (gốc 13.3 MB)


⚠️ Cột `muc_bien_dong` không khớp hoàn toàn: `pd.cut` sinh ra `category` **có
thứ tự** (thấp < vừa < cao < rất cao), còn `astype("category")` khi nạp lại chỉ
tạo category **không thứ tự**. Bản đồ kiểu dạng chuỗi không mang được thông tin
đó — thêm một lý do để dùng parquet khi có thể.

In [19]:
print(f"gốc     : {san_sang['muc_bien_dong'].dtype}")
print(f"phục hồi: {phuc_hoi['muc_bien_dong'].dtype}")
print(f"\ncó thứ tự? gốc={san_sang['muc_bien_dong'].cat.ordered} · phục hồi={phuc_hoi['muc_bien_dong'].cat.ordered}")
print("→ so sánh `>` chạy được trên bản gốc, ném lỗi trên bản phục hồi")

gốc     : category
phục hồi: category

có thứ tự? gốc=True · phục hồi=False
→ so sánh `>` chạy được trên bản gốc, ném lỗi trên bản phục hồi


## 7 · Kết quả cuối

Đặt hai frame cạnh nhau: thô từ API, và sau khi qua `chuan_bi_du_lieu()`.

In [20]:
tong_ket = pd.concat(
    [
        kiem_tra_chat_luong(tho).assign(giai_doan="thô từ API"),
        kiem_tra_chat_luong(san_sang).assign(giai_doan="sau chuẩn bị"),
    ]
).set_index("giai_doan")
tong_ket

,dòng,mã,phiên đầu,phiên cuối,ô thiếu,dòng trùng khoá,giá trị âm,MB
giai_doan,,,,,,,,
thô từ API,97625,199,2024-08-12,2026-08-12,0,0,0,9.3
sau chuẩn bị,97625,199,2024-08-12,2026-08-12,8140,0,41226,13.3


In [21]:
chung = [c for c in tho.columns if c in san_sang.columns]
mb_tho = tho[chung].memory_usage(deep=True).sum() / 1024**2
mb_sau = san_sang[chung].memory_usage(deep=True).sum() / 1024**2

print(f"Số cột : {tho.shape[1]} → {san_sang.shape[1]}  (+{san_sang.shape[1] - tho.shape[1]} cột phái sinh)")
print(f"Thời gian: {t_chuan_bi:.1f} giây cho {len(san_sang):,} dòng")
print()
print("Bộ nhớ — so trên CÙNG BỘ CỘT gốc:")
print(f"  {mb_tho:>5.1f} MB → {mb_sau:>5.1f} MB   (nhẹ hơn {mb_tho / mb_sau:.1f} lần)")
print()
print("Bộ nhớ toàn frame:")
print(f"  {tho.memory_usage(deep=True).sum() / 1024**2:>5.1f} MB → "
      f"{san_sang.memory_usage(deep=True).sum() / 1024**2:>5.1f} MB   ← LỚN hơn, vì có thêm 7 cột")

Số cột : 7 → 14  (+7 cột phái sinh)
Thời gian: 17.2 giây cho 97,625 dòng

Bộ nhớ — so trên CÙNG BỘ CỘT gốc:
    9.3 MB →   3.2 MB   (nhẹ hơn 2.9 lần)

Bộ nhớ toàn frame:
    9.3 MB →  13.3 MB   ← LỚN hơn, vì có thêm 7 cột


### Dùng ngay được

Frame đã sẵn sàng cho phân tích, không cần bước chuẩn bị nào nữa:

In [22]:
phien_cuoi = san_sang["date"].max()
hom_nay_bang = san_sang[san_sang["date"] == phien_cuoi]

print(f"Phiên {phien_cuoi:%d/%m/%Y} — {len(hom_nay_bang)} mã\n")
print("Đột biến khối lượng (≥ 2× bình quân 20 phiên):")
dot_bien = hom_nay_bang[hom_nay_bang["so_lan_kl"] >= 2].nlargest(8, "so_lan_kl")
print(
    dot_bien[["symbol", "close", "ls", "so_lan_kl", "muc_bien_dong"]]
    .round({"close": 2, "ls": 2, "so_lan_kl": 1})
    .to_string(index=False)
)

Phiên 12/08/2026 — 198 mã

Đột biến khối lượng (≥ 2× bình quân 20 phiên):
symbol     close    ls  so_lan_kl muc_bien_dong
   L10 25.400000  6.95        6.1          thấp
   KSB 14.250000  6.74        4.7       rất cao
   ABR 13.500000  6.72        4.6           vừa
   ANT 18.549999 -6.78        4.1       rất cao


⚠️ Để ý cột `close` in ra `18.549999` thay vì `18.55`. Đó chính là `float32` từ
bước hai lộ diện — `.round(2)` làm tròn *giá trị*, nhưng `float32` không biểu
diễn nổi `18.55` một cách chính xác nên phần dư hiện ra khi in.

Sai lệch cỡ `10⁻⁵` nghìn VND, nhỏ hơn bước giá hàng nghìn lần, nên **phép tính
vẫn đúng**. Nhưng khi *hiển thị* cho người đọc thì ép về `float64` trước:

In [23]:
print(
    dot_bien[["symbol", "close", "ls", "so_lan_kl"]]
    .astype({"close": "float64", "ls": "float64"})
    .round(2)
    .to_string(index=False)
)

symbol  close    ls  so_lan_kl
   L10  25.40  6.95       6.14
   KSB  14.25  6.74       4.68
   ABR  13.50  6.72       4.59
   ANT  18.55 -6.78       4.12


In [24]:
print("Phân bố hướng và mức biến động phiên gần nhất:")
print(pd.crosstab(hom_nay_bang["muc_bien_dong"], hom_nay_bang["huong"]).to_string())

Phân bố hướng và mức biến động phiên gần nhất:


huong          giảm  tăng  đứng giá
muc_bien_dong                      
thấp             23    31        55
vừa              18    45        12
cao               2     5         0
rất cao           3     3         1


## Tổng kết

| Hàm | Việc |
|---|---|
| `kiem_tra_chat_luong(df)` | báo cáo một dòng: số dòng, mã, ô thiếu, trùng khoá, bộ nhớ |
| `kiem_tra_ohlcv(df)` | ba bất biến của một thanh giá, trả về các dòng vi phạm |
| `toi_uu_kieu(df)` | `category` + `float32` + `int64`, trả frame mới |
| `them_cot_phai_sinh(df)` | GTGD, lợi suất, biên độ, đột biến khối lượng — qua `groupby` |
| `chuan_bi_du_lieu(...)` | gộp cả bốn, một lời gọi |

**Năm điều notebook này chứng minh:**

1. **`duplicated()` phải nêu `subset` khoá thật.** So cả hàng thì một mã tải hai
   lần với giá đã điều chỉnh khác nhau sẽ lọt qua.
2. **`float32` an toàn cho giá cổ phiếu Việt Nam** — sai lệch nhỏ hơn bước giá
   hàng nghìn lần — nhưng **không an toàn cho giá trị giao dịch** vì GTGD vượt
   xa 7 chữ số có nghĩa của nó.
3. **Mọi phép theo thời gian phải qua `groupby("symbol")`.** Notebook khẳng
   định điều đó bằng `assert`, và đo lợi suất bịa mà cách sai tạo ra.
4. **Hàm kiểm tra phải được kiểm tra.** Notebook bẻ hỏng dữ liệu để chứng minh
   `AssertionError` thật sự được ném.
5. **CSV quên hết kiểu dữ liệu.** Frame phình lên nhiều lần khi nạp lại, `date`
   thành chuỗi, và `category` có thứ tự mất thứ tự — parquet giải quyết trọn vẹn.

---

**Hết nhóm cơ bản.** Bốn notebook `51`–`54` đã dựng nền và cho một sản phẩm dùng
được. Nhóm trung cấp bắt đầu ở `55_groupby` — split-apply-combine, thứ mà mini
project này mới chỉ chạm tới bề mặt.